<a href="https://colab.research.google.com/github/ayaz-ali-36/Machine-Learning/blob/main/Hyperparameter%20Tunning/lab_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hyperparameters Tunning

In [32]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import f1_score

import optuna
import warnings
warnings.filterwarnings("ignore")

In [33]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'

cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
        'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']

df = pd.read_csv(url, names=cols)

In [34]:
zero_cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

for col in zero_cols:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

In [35]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [36]:
scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [38]:
models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier()
}

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    print(name, f1_score(y_test, preds))

Logistic 0.5454545454545454
RandomForest 0.5961538461538461
KNN 0.6346153846153846


Grid Search Hyperparmeters tunnimg (LR)

In [39]:
grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['liblinear']
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid=grid,
    cv=5,
    scoring='f1'
)

lr_grid.fit(X_train_sc, y_train)

print(lr_grid.best_params_)

{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}


Random Search Hyperparmeters tunnimg (RF)

In [41]:
rf_grid = {
    'n_estimators': [50,100,200],
    'max_depth': [3,5,10]
}

rf_random = RandomizedSearchCV(
    RandomForestClassifier(),
    param_distributions=rf_grid,
    n_iter=5,
    cv=5,
    scoring='f1',
    random_state=42
)

rf_random.fit(X_train, y_train)

print(rf_random.best_params_)

{'n_estimators': 100, 'max_depth': 10}


GridSerach (KNN)

In [43]:
knn_grid = {
    'n_neighbors': [3,5,7,9]
}

knn_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=knn_grid,
    cv=5,
    scoring='f1'
)

knn_search.fit(X_train_sc, y_train)

print(knn_search.best_params_)

{'n_neighbors': 9}


In [44]:
models = {
    "Logistic": lr_grid.best_estimator_,
    "RandomForest": rf_random.best_estimator_,
    "KNN": knn_search.best_estimator_
}

for name, model in models.items():
    preds = model.predict(X_test_sc)
    print(name, f1_score(y_test, preds))

Logistic 0.5490196078431373
RandomForest 0.0
KNN 0.6226415094339622
